# Projet SIEM : Suricata, Filebeat, Elasticsearch

## Objectifs

Ce projet vous permettra de :
- Vérifier le bon fonctionnement de la stack SIEM
- Analyser les données indexées par Filebeat (Suricata)
- Détecter des anomalies sans ML
- Détecter des anomalies avec ML (Isolation Forest)

## Modalité
- groupe de 3 à 4
- output : projet git avec ce notebook détaillé et complété

## Architecture SIEM

```
Suricata   →   Filebeat   →   Elasticsearch   →   Kibana
(IDS/HIDS)   (Collecteur)      (Stockage)   (Visualisation/Alerting)
```

## Prérequis

1. Démarrer la stack :
```bash
docker-compose -f docker-compose-siem.yml up -d
```

2. Attendre quelques minutes que Suricata génère des logs et que Filebeat les indexe dans Elasticsearch.

In [5]:
# Configuration et connexion à Elasticsearch
from elasticsearch import Elasticsearch
from datetime import datetime, timedelta
import json
import subprocess
import os
import warnings
from urllib3.exceptions import InsecureRequestWarning

warnings.simplefilter("ignore", InsecureRequestWarning)
# Configuration
ES_HOST = "https://localhost:9200"
ES_USER = "elastic"
ES_PASSWORD = "changeme"  # Modifiez selon votre .env

# Connexion
es = Elasticsearch(
    [ES_HOST],
    basic_auth=(ES_USER, ES_PASSWORD),
    verify_certs=False
)

# Vérification de la connexion
health = es.cluster.health()
print(f"✅ Cluster Elasticsearch: {health['status']} ({health['number_of_nodes']} nœuds)")

✅ Cluster Elasticsearch: green (3 nœuds)


C:\Users\gohau\Dev\Projet ELK\elasticsearch_cookbook\.venv\Lib\site-packages\elasticsearch\_sync\client\__init__.py:313: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


## 1. Vérification de la stack

In [6]:
# Vérification des services Docker
services = ['es01', 'es02', 'es03', 'kibana', 'suricata', 'filebeat']
running = []

for service in services:
    try:
        result = subprocess.run(
            ['docker', 'ps', '--filter', f'name={service}', '--format', '{{.Names}}'],
            capture_output=True, text=True, timeout=5
        )
        if service in result.stdout:
            running.append(service)
            print(f"✅ {service}")
        else:
            print(f"❌ {service}")
    except:
        print(f"❌ {service}")

if len(running) == len(services):
    print(f"\n✅ Tous les services sont démarrés ({len(running)}/{len(services)})")
else:
    print(f"\n⚠️  Services démarrés: {len(running)}/{len(services)}")

✅ es01
✅ es02
✅ es03
✅ kibana
✅ suricata
✅ filebeat

✅ Tous les services sont démarrés (6/6)


## 2. Vérification de l'injection des données

In [8]:
# Recherche des index Suricata
def get_suricata_index():
    """Retourne le nom de l'index Suricata le plus récent"""
    try:
        indices = es.indices.get(index="suricata-*")
        if indices:
            return sorted(indices.keys())[-1]
    except:
        pass
    return "suricata-*"

index_name = get_suricata_index()

# Comptage des documents
try:
    count = es.count(index=index_name)
    print(f"📊 Index: {index_name}")
    print(f"📈 Nombre de documents: {count['count']:,}")
    
    # Exemple de document
    if count['count'] > 0:
        sample = es.search(index=index_name, size=1, query={"match_all": {}})
        if sample['hits']['hits']:
            doc = sample['hits']['hits'][0]['_source']
            print(f"\n📄 Exemple de document:")
            print(f"   Type: {doc.get('event_type', 'N/A')}")
            print(f"   Timestamp: {doc.get('@timestamp', doc.get('timestamp', 'N/A'))}")
            if 'src_ip' in doc:
                print(f"   Source: {doc.get('src_ip')}:{doc.get('src_port', 'N/A')}")
                print(f"   Destination: {doc.get('dest_ip')}:{doc.get('dest_port', 'N/A')}")
            if 'alert' in doc:
                alert = doc['alert']
                print(f"   Alerte: {alert.get('signature', 'N/A')}")
                print(f"   Sévérité: {alert.get('severity', 'N/A')}")
except Exception as e:
    print(f"❌ Erreur: {e}")

📊 Index: .ds-suricata-2026.03.09-2026.03.09-000001
📈 Nombre de documents: 125,995

📄 Exemple de document:
   Type: flow
   Timestamp: 2026-03-09T22:20:17.311Z
   Source: 192.168.65.1:45196
   Destination: 192.168.65.7:2375


## 4. Simuler des comportements anormaux


#### Simulation de scans suspects (en ligne de commande ou en python) 

In [9]:
import socket
import socket
from datetime import datetime

def port_scanner(target, port_range):
    print(f"Lancement du scan sur {target}...")
    print(f"Début : {datetime.now().strftime('%H:%M:%S')}")
    
    detected_ports = []
    
    for port in port_range:
        # Création d'une socket TCP
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(0.1) # Timeout court pour aller vite
        
        # On tente la connexion
        result = sock.connect_ex((target, port))
        
        # Si le port est ouvert (0), on le note
        if result == 0:
            detected_ports.append(port)
            print(f"Port {port}: OUVERT")
            
        sock.close()

    print(f"\n Scan terminé à {datetime.now().strftime('%H:%M:%S')}")
    print(f" Ports ouverts trouvés : {detected_ports}")
    print("\n Attendez 30s et vérifiez votre index Suricata dans Elasticsearch.")

# Configuration du scan
TARGET_IP = "127.0.0.1"  # On scanne l'hôte local (Docker Desktop)
PORTS_TO_SCAN = range(1, 1025) # Scan des ports "Well-Known" (1 à 1024)

# Exécution
port_scanner(TARGET_IP, PORTS_TO_SCAN)

Lancement du scan de ports sur 192.168.65.3...
Scan terminé.


#### Simulation de burst HTTP en ligne de commande (en ligne de commande ou en python) 

In [11]:
import requests
import time

def simulate_http_burst(target_url="http://192.168.65.3:80", num_requests=500):
    print(f":rocket: Lancement du burst HTTP vers {target_url}...")
    for i in range(num_requests):
        try:
            requests.get(target_url, timeout=0.5)
        except requests.exceptions.RequestException:
            pass
    print(f":white_check_mark: Burst de {num_requests} requêtes terminé.")

simulate_http_burst()

:rocket: Lancement du burst HTTP vers http://192.168.65.3:80...
:white_check_mark: Burst de 500 requêtes terminé.


## 4. Détection d'anomalies sans ML

L'objectif est de construire des règles qui détectent vos simulations de comportements précédents.

Un exemple de détection d'anomalies basiques basées sur des règles:


In [6]:
def detect_anomalies_basic():
    """Détecte des anomalies sans ML"""
    anomalies = []
    
    # 1. IPs sources avec beaucoup d'alertes différentes (scan suspect)
    query = {
        "size": 0,
        "query": {"term": {"event_type": "alert"}},
        "aggs": {
            "suspicious_ips": {
                "terms": {"field": "src_ip", "size": 10},
                "aggs": {
                    "unique_signatures": {"cardinality": {"field": "alert.signature_id"}},
                    "unique_dest_ips": {"cardinality": {"field": "dest_ip"}},
                    "total_alerts": {"value_count": {"field": "event_type"}}
                }
            }
        }
    }
    
    result = es.search(index=index_name, body=query)
    
    print("Détection d'anomalies (règles basiques)\n")

    for bucket in result['aggregations']['suspicious_ips']['buckets']:
        ip = bucket['key']
        unique_sigs = bucket['unique_signatures']['value']
        unique_dests = bucket['unique_dest_ips']['value']
        total = bucket['total_alerts']['value']
        
        # Critères d'anomalie
        if unique_sigs > 3 or unique_dests > 5:
            anomalies.append({
                "ip": ip,
                "type": "Scan suspect",
                "signatures": unique_sigs,
                "destinations": unique_dests,
                "total": total
            })
            print(f"🚨 {ip}: {unique_sigs} signatures, {unique_dests} destinations ({total} alertes)")

    return anomalies

anomalies = detect_anomalies_basic()
if not anomalies:
    print("\nAucune anomalie détectée avec les règles basiques")

Détection d'anomalies (règles basiques)


Aucune anomalie détectée avec les règles basiques


## 5. Détection d'anomalies avec ML

L'objectif est d'à partir de données labelisées comme étant anomarles ou non, construire un jeu de donnée et entrainer un algorithme de machine learning (classifier binaire) sur ces données. Le modèle devra ensuite etre appelé pour prédire les futurs comportements anormaux.

Indice: le modèle Isolation Forest pourrait etre utile

In [43]:
# TODO

## Bonus
- améliorer la stack (ex: ajout de Wazuh)
- dashboard Kibana pour voir en live les simulations de comportements anormaux et les détection d'anomalie (timeline des alertes, etc.)
- analyse statistique avancée
- simulations de comportement anormaux avancée
- utilisation du module de détection d'anomalie d'Elasticsearch (https://www.elastic.co/docs/explore-analyze/machine-learning/anomaly-detection)
- collaboration en groupe sur le projet Git (Pull requests, commits, etc.)
- utilisation de docker / docker compose / devcontainer
- etc.

##import requests
import time
import random

# L'URL cible : Comme Suricata est en mode host sur votre machine Windows, 
# n'importe quelle requête sortante ou locale devrait être vue par l'IDS.
# On peut viser localhost ou une IP externe connue.
TARGET_URL = "http://localhost:5601" # On vise Kibana pour le test

# Liste de "payloads" typiques qui déclenchent des alertes IDS
SUSPICIOUS_PATHS = [
    "/etc/passwd",
    "/.git/config",
    "/wp-admin.php",
    "/admin/config.php",
    "/.env",
    "/shell.php",
    "/sql?query=SELECT * FROM users--"
]

def simulate_scan():
    print(f"🚀 Démarrage de la simulation sur {TARGET_URL}...")
    
    for path in SUSPICIOUS_PATHS:
        url = f"{TARGET_URL}{path}"
        try:
            print(f"🔍 Test du chemin suspect : {path}", end=" -> ")
            # On envoie une requête avec un User-Agent suspect pour certains IDS
            headers = {'User-Agent': 'Mozilla/5.0 (compatible; Nmap Scripting Engine)'}
            response = requests.get(url, headers=headers, timeout=2)
            print(f"Réponse: {response.status_code}")
        except Exception as e:
            print(f"Note: Requête envoyée (le service cible n'a pas besoin d'être actif pour que Suricata voit le trafic)")
        
        # Petit délai pour éviter de saturer si on boucle
        time.sleep(0.5)

    print("\n✅ Simulation terminée. Attendez 30s que Filebeat traite les logs.")

# Exécuter la simulation
simulate_scan()

In [1]:
## test de fuzing
import requests
import time
import random

# L'URL cible : Comme Suricata est en mode host sur votre machine Windows, 
# n'importe quelle requête sortante ou locale devrait être vue par l'IDS.
# On peut viser localhost ou une IP externe connue.
TARGET_URL = "http://localhost:5601" # On vise Kibana pour le test

# Liste de "payloads" typiques qui déclenchent des alertes IDS
SUSPICIOUS_PATHS = [
    "/etc/passwd",
    "/.git/config",
    "/wp-admin.php",
    "/admin/config.php",
    "/.env",
    "/shell.php",
    "/sql?query=SELECT * FROM users--"
]

def simulate_scan():
    print(f"🚀 Démarrage de la simulation sur {TARGET_URL}...")
    
    for path in SUSPICIOUS_PATHS:
        url = f"{TARGET_URL}{path}"
        try:
            print(f"🔍 Test du chemin suspect : {path}", end=" -> ")
            # On envoie une requête avec un User-Agent suspect pour certains IDS
            headers = {'User-Agent': 'Mozilla/5.0 (compatible; Nmap Scripting Engine)'}
            response = requests.get(url, headers=headers, timeout=2)
            print(f"Réponse: {response.status_code}")
        except Exception as e:
            print(f"Note: Requête envoyée (le service cible n'a pas besoin d'être actif pour que Suricata voit le trafic)")
        
        # Petit délai pour éviter de saturer si on boucle
        time.sleep(0.5)

    print("\n✅ Simulation terminée. Attendez 30s que Filebeat traite les logs.")

# Exécuter la simulation
simulate_scan()

🚀 Démarrage de la simulation sur http://localhost:5601...
🔍 Test du chemin suspect : /etc/passwd -> Note: Requête envoyée (le service cible n'a pas besoin d'être actif pour que Suricata voit le trafic)
🔍 Test du chemin suspect : /.git/config -> 

KeyboardInterrupt: 